In [1]:
import vllm
import tqdm as notebook_tqdm
import dotenv
import os

In [2]:
dotenv.load_dotenv(dotenv.find_dotenv())


True

In [3]:
# !export VLLM_ATTENTION_BACKEND=XFORMERS
# !export VLLM_ATTENTION_BACKEND=TORCH
os.environ["VLLM_USE_FLASHINFER"] = "0"
os.environ["VLLM_USE_FLASHINFER_SAMPLER"] = "0"

# 2. Force the attention backend away from FlashAttention-2 / FlashInfer
# Note: "TRITON" is often faster than "TORCH" on Turing cards if available, 
# but "TORCH" is the safest fallback if Triton throws errors.
os.environ["VLLM_ATTENTION_BACKEND"] = "TORCH" 

# 3. Suppress custom built workspace allocations that might trigger JIT compilation
os.environ["VLLM_USE_TRITON_FLASH_ATTN"] = "0"

In [4]:
from vllm import LLM, SamplingParams

# Define the model and enforce a capped context limit to protect VRAM
model_id = "Qwen/Qwen3.5-2B"
model_id = "Qwen/Qwen3-0.6B"
llm = LLM(
    model=model_id,
    max_model_len=2048, # Dropped from 4096 to prevent immediate VRAM OOM on a 4GB card
    dtype="float16",    # Explicitly avoid any default bfloat16 parsing
    # enforce_eager=True, # Disables CUDA graph / JIT engine overhead that crashes old architectures
    hf_token=os.getenv("HUGGING_FACE_API_KEY"),
    gpu_memory_utilization=0.70
)

# Configure your generation parameters
sampling_params = SamplingParams(
    temperature=0.7,
    top_p=0.9,
    max_tokens=512
)

# Prepare prompts using Qwen's chat template structure
prompts = [
    "<|im_start|>user\nWrite a python function to validate an email address.<|im_end|>\n<|im_start|>assistant\n"
]

# Run inference
outputs = llm.generate(prompts, sampling_params)

# Print the results
for output in outputs:
    print(output.outputs[0].text)

/home/hex/Documents/devops/dev-bits-daily/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


INFO 05-23 17:06:56 [utils.py:240] non-default args: {'dtype': 'float16', 'max_model_len': 2048, 'gpu_memory_utilization': 0.7, 'disable_log_stats': True, 'hf_token': 'hf_***'}
WARNING 05-23 17:06:56 [envs.py:1866] Unknown vLLM environment variable detected: VLLM_USE_FLASHINFER
WARNING 05-23 17:06:56 [envs.py:1866] Unknown vLLM environment variable detected: VLLM_ATTENTION_BACKEND
WARNING 05-23 17:06:56 [envs.py:1866] Unknown vLLM environment variable detected: VLLM_USE_TRITON_FLASH_ATTN
INFO 05-23 17:06:59 [model.py:568] Resolved architecture: Qwen3ForCausalLM
WARNING 05-23 17:06:59 [model.py:2035] Casting torch.bfloat16 to torch.float16.
INFO 05-23 17:06:59 [model.py:1697] Using max model len 2048
INFO 05-23 17:06:59 [scheduler.py:239] Chunked prefill is enabled with max_num_batched_tokens=8192.
INFO 05-23 17:06:59 [vllm.py:886] Asynchronous scheduling is enabled.
INFO 05-23 17:06:59 [kernel.py:212] Final IR op priority after setting platform defaults: IrOpPriorityConfig(rms_norm=['n

(EngineCore pid=303468) INFO 05-23 17:07:04 [core.py:109] Initializing a V1 LLM engine (v0.21.0) with config: model='Qwen/Qwen3-0.6B', speculative_config=None, tokenizer='Qwen/Qwen3-0.6B', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, tokenizer_revision=None, trust_remote_code=False, dtype=torch.float16, max_seq_len=2048, download_dir=None, load_format=auto, tensor_parallel_size=1, pipeline_parallel_size=1, data_parallel_size=1, decode_context_parallel_size=1, dcp_comm_backend=ag_rs, disable_custom_all_reduce=False, quantization=None, quantization_config=None, enforce_eager=False, enable_return_routed_experts=False, kv_cache_dtype=auto, device_config=cuda, structured_outputs_config=StructuredOutputsConfig(backend='auto', disable_any_whitespace=False, disable_additional_properties=False, reasoning_parser='', reasoning_parser_plugin='', enable_in_reasoning=False), observability_config=ObservabilityConfig(show_hidden_metrics_for_version=None, otlp_traces_endpoint=None, co

Loading safetensors checkpoint shards:   0% Completed | 0/1 [00:00<?, ?it/s]
Loading safetensors checkpoint shards: 100% Completed | 1/1 [00:00<00:00,  2.16it/s]
Loading safetensors checkpoint shards: 100% Completed | 1/1 [00:00<00:00,  2.15it/s]
(EngineCore pid=303468) 


(EngineCore pid=303468) INFO 05-23 17:07:08 [default_loader.py:397] Loading weights took 0.49 seconds
(EngineCore pid=303468) INFO 05-23 17:07:08 [gpu_model_runner.py:4959] Model loading took 1.12 GiB memory and 2.397341 seconds
(EngineCore pid=303468) INFO 05-23 17:07:17 [backends.py:1089] Using cache directory: /home/hex/.cache/vllm/torch_compile_cache/b60368b542/rank_0_0/backbone for vLLM's torch.compile
(EngineCore pid=303468) INFO 05-23 17:07:17 [backends.py:1148] Dynamo bytecode transform time: 8.37 s


(EngineCore pid=303468) [rank0]:W0523 17:07:19.342000 303468 torch/_inductor/utils.py:1731] Not enough SMs to use max_autotune_gemm mode


(EngineCore pid=303468) INFO 05-23 17:07:25 [backends.py:378] Cache the graph of compile range (1, 8192) for later use
(EngineCore pid=303468) INFO 05-23 17:07:31 [backends.py:393] Compiling a graph for compile range (1, 8192) takes 13.29 s
(EngineCore pid=303468) INFO 05-23 17:07:35 [decorators.py:708] saved AOT compiled function to /home/hex/.cache/vllm/torch_compile_cache/torch_aot_compile/c381234dbc11584b6e7eda140b037b8c425cee6c827719bf784bffec3e5f10e5/rank_0_0/model
(EngineCore pid=303468) INFO 05-23 17:07:35 [monitor.py:53] torch.compile took 26.44 s in total
(EngineCore pid=303468) INFO 05-23 17:07:37 [monitor.py:81] Initial profiling/warmup run took 1.48 s
(EngineCore pid=303468) INFO 05-23 17:08:05 [gpu_model_runner.py:6063] Profiling CUDA graph memory: PIECEWISE=51 (largest=512), FULL=35 (largest=256)
(EngineCore pid=303468) INFO 05-23 17:08:10 [gpu_model_runner.py:6142] Estimated CUDA graph memory: 0.83 GiB total
(EngineCore pid=303468) INFO 05-23 17:08:10 [gpu_worker.py:462

(EngineCore pid=303468) Process EngineCore:
(EngineCore pid=303468) Traceback (most recent call last):
(EngineCore pid=303468)   File "/usr/lib/python3.12/multiprocessing/process.py", line 314, in _bootstrap
(EngineCore pid=303468)     self.run()
(EngineCore pid=303468)   File "/usr/lib/python3.12/multiprocessing/process.py", line 108, in run
(EngineCore pid=303468)     self._target(*self._args, **self._kwargs)
(EngineCore pid=303468)   File "/home/hex/Documents/devops/dev-bits-daily/.venv/lib/python3.12/site-packages/vllm/v1/engine/core.py", line 1144, in run_engine_core
(EngineCore pid=303468)     raise e
(EngineCore pid=303468)   File "/home/hex/Documents/devops/dev-bits-daily/.venv/lib/python3.12/site-packages/vllm/v1/engine/core.py", line 1114, in run_engine_core
(EngineCore pid=303468)     engine_core = EngineCoreProc(*args, engine_index=dp_rank, **kwargs)
(EngineCore pid=303468)                   ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
(EngineCore pid=303468)   Fil

RuntimeError: Engine core initialization failed. See root cause above. Failed core proc(s): {}

In [5]:
import os
# Ensure the environment variables are set before importing vLLM
os.environ["VLLM_TARGET_DEVICE"] = "cpu"
os.environ["VLLM_MEMORY_PROFILER_ESTIMATE_CUDAGRAPHS"] = "0"

from vllm import EngineArgs, LLMEngine

engine_args = EngineArgs(
    model="Qwen/Qwen3-0.6B",
    device="cpu",        # Explicitly target system RAM/CPU
    dtype="bfloat16",    # Best datatype for modern CPU instruction sets
    max_model_len=2048
)

# Initialize your engine normally from here...

TypeError: EngineArgs.__init__() got an unexpected keyword argument 'device'

In [ ]:
import os

# 1. Force vLLM to target the CPU backend via environment variables
os.environ["VLLM_TARGET_DEVICE"] = "cpu"
os.environ["VLLM_MEMORY_PROFILER_ESTIMATE_CUDAGRAPHS"] = "0"

from vllm import EngineArgs, LLMEngine

# 2. Initialize EngineArgs without the 'device' parameter
engine_args = EngineArgs(
    model="Qwen/Qwen3-0.6B",
    dtype="float16",  # Excellent choice for modern AVX-512 / AMX instruction sets
)

# 3. Instantiate your engine
engine = LLMEngine.from_engine_args(engine_args)

/home/hex/Documents/devops/dev-bits-daily/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


INFO 05-23 17:13:53 [model.py:568] Resolved architecture: Qwen3ForCausalLM
WARNING 05-23 17:13:53 [model.py:2035] Casting torch.bfloat16 to torch.float16.
INFO 05-23 17:13:53 [model.py:1697] Using max model len 40960
INFO 05-23 17:13:53 [scheduler.py:239] Chunked prefill is enabled with max_num_batched_tokens=2048.
INFO 05-23 17:13:53 [vllm.py:886] Asynchronous scheduling is enabled.
INFO 05-23 17:13:53 [kernel.py:212] Final IR op priority after setting platform defaults: IrOpPriorityConfig(rms_norm=['native'], fused_add_rms_norm=['native'])
(EngineCore pid=314301) INFO 05-23 17:13:58 [core.py:109] Initializing a V1 LLM engine (v0.21.0) with config: model='Qwen/Qwen3-0.6B', speculative_config=None, tokenizer='Qwen/Qwen3-0.6B', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, tokenizer_revision=None, trust_remote_code=False, dtype=torch.float16, max_seq_len=40960, download_dir=None, load_format=auto, tensor_parallel_size=1, pipeline_parallel_size=1, data_parallel_size=1, d

Loading safetensors checkpoint shards:   0% Completed | 0/1 [00:00<?, ?it/s]
Loading safetensors checkpoint shards: 100% Completed | 1/1 [00:00<00:00,  2.09it/s]
Loading safetensors checkpoint shards: 100% Completed | 1/1 [00:00<00:00,  2.08it/s]
(EngineCore pid=314301) 


(EngineCore pid=314301) INFO 05-23 17:14:02 [default_loader.py:397] Loading weights took 0.51 seconds
(EngineCore pid=314301) INFO 05-23 17:14:02 [gpu_model_runner.py:4959] Model loading took 1.12 GiB memory and 2.098474 seconds
(EngineCore pid=314301) INFO 05-23 17:14:11 [backends.py:1089] Using cache directory: /home/hex/.cache/vllm/torch_compile_cache/a6b7d9a716/rank_0_0/backbone for vLLM's torch.compile
(EngineCore pid=314301) INFO 05-23 17:14:11 [backends.py:1148] Dynamo bytecode transform time: 8.53 s


(EngineCore pid=314301) Process EngineCore:
(EngineCore pid=314301) Traceback (most recent call last):
(EngineCore pid=314301)   File "/usr/lib/python3.12/multiprocessing/process.py", line 314, in _bootstrap
(EngineCore pid=314301)     self.run()
(EngineCore pid=314301)   File "/usr/lib/python3.12/multiprocessing/process.py", line 108, in run
(EngineCore pid=314301)     self._target(*self._args, **self._kwargs)
(EngineCore pid=314301)   File "/home/hex/Documents/devops/dev-bits-daily/.venv/lib/python3.12/site-packages/vllm/v1/engine/core.py", line 1114, in run_engine_core
(EngineCore pid=314301)     engine_core = EngineCoreProc(*args, engine_index=dp_rank, **kwargs)
(EngineCore pid=314301)                   ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
(EngineCore pid=314301)   File "/home/hex/Documents/devops/dev-bits-daily/.venv/lib/python3.12/site-packages/vllm/tracing/otel.py", line 178, in sync_wrapper
(EngineCore pid=314301)     return func(*args, **kwargs)
(EngineCore pi